# CoT기법을 활용하여 문제 해결하기

### 1. 준비 단계

In [14]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

In [15]:
from huggingface_hub import login
login()

In [16]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_pipe(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return pipeline("text-generation", model=model, tokenizer=tok)

def free(name):
    globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()

### 2. 문제 정의하기

In [17]:
# 문제 1
Q1 = """서점에서 책 한 권 가격은 18,000원이다.
행사로 3권을 사면 총 가격에서 10% 할인이 적용된다.
예를 들어 7권을 사면 6권까지는 3권씩 묶음 할인이 적용되고 나머지 한 권에는 적용되지 않는다.

철수는 책 11권을 샀다.
그리고 결제 금액의 5%를 포인트로 적립받았다.

철수가 실제로 지불한 금액과 적립된 포인트 금액은 얼마인가?"""

# 실제 지불 금액: 181,800원
# 적립 포인트: 9,090원

# 문제 2
Q2 = """상자에 빨간 공 3개, 파란 공 2개가 있다.
영희는 공 하나를 무작위로 뽑았고,
영희는 파란 공을 뽑았지만 색을 확인하지 않고 주머니에 넣었다.

영희가 다시 공 하나를 뽑는다.
두 번째로 뽑은 공이 빨간 공일 확률은 얼마인가?"""

# 3/4 ( = 0.75 = 75% )


### 3. 기존 Instruct 모델

In [24]:
# 모델 로드
instruct = load_pipe("meta-llama/Llama-3.1-8B-Instruct")

questions = [Q1, Q2]

system_prompt = "당신은 논리적 추론과 계산에 강한 한국어 AI 어시스턴트입니다. 문제를 단계적으로 분석하고, 조건을 정리한 뒤, 수식과 계산 과정을 포함하여 정확한 답을 도출하세요."

answers = []

# 실행
for i, q in enumerate(questions, 1):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q},
    ]

    out = instruct(
        messages,
        max_new_tokens=768,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

    try:
        ans = out[0]["generated_text"][-1]["content"]
    except:
        ans = out[0]["generated_text"]

    answers.append(ans)

    print(f"\n===== Q{i} =====")
    print("문제:")
    print(q)
    print("\n답변:")
    print(ans)
    print("================\n")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== Q1 =====
문제:
서점에서 책 한 권 가격은 18,000원이다.
행사로 3권을 사면 총 가격에서 10% 할인이 적용된다.
예를 들어 7권을 사면 6권까지는 3권씩 묶음 할인이 적용되고 나머지 한 권에는 적용되지 않는다.

철수는 책 11권을 샀다.
그리고 결제 금액의 5%를 포인트로 적립받았다.

철수가 실제로 지불한 금액과 적립된 포인트 금액은 얼마인가?

답변:
먼저, 책 11권을 샀을 때 3권씩 묶음으로 3세트를 구매한 경우와 나머지 2권을 구매한 경우를 구분해야 합니다.

11권을 3세트로 구매한 경우, 3세트는 3 * 18,000원 = 54,000원입니다. 나머지 2권은 2 * 18,000원 = 36,000원입니다. 총 가격은 54,000원 + 36,000원 = 90,000원입니다.

할인은 3세트에만 적용되므로, 90,000원에서 10% 할인을 적용하면 90,000원 * 0.1 = 9,000원이 할인됩니다. 총 할인된 가격은 90,000원 - 9,000원 = 81,000원입니다.

나머지 2권에는 할인이 적용되지 않으므로, 실제로 지불한 금액은 36,000원 + 81,000원 = 117,000원입니다.

철수가 구매한 금액의 5%를 포인트로 적립받았다면, 포인트 금액은 117,000원 * 0.05 = 5,850원입니다.

결과적으로, 철수가 실제로 지불한 금액은 117,000원이고, 적립된 포인트 금액은 5,850원입니다.


===== Q2 =====
문제:
상자에 빨간 공 3개, 파란 공 2개가 있다.
영희는 공 하나를 무작위로 뽑았고,
영희는 파란 공을 뽑았지만 색을 확인하지 않고 주머니에 넣었다.

영희가 다시 공 하나를 뽑는다.
두 번째로 뽑은 공이 빨간 공일 확률은 얼마인가?

답변:
문제를 단계적으로 분석해 보겠습니다.

1. 상자에 빨간 공 3개, 파란 공 2개가 있다.
2. 영희는 공 하나를 무작위로 뽑았는데, 파란 공을 뽑았다.
3. 영희는 색을 확인하지 않고 주머니에 넣었다.

이제, 두 번째로 뽑은 공이 빨간 

### 4. CoT 방식 선택

- Zero-shot CoT : 예시 없이, 간단한 지시만으로 추론 과정을 유도
- Few-shot CoT : 몇 개의 풀이 예시를 보고 그 패턴을 따라해서 추론
- Self-Consistency CoT : 여러 추론 경로를 생성한 뒤 다수결로 결정
- Structured CoT Prompt : 조건 정리→식 세우기→계산 등 단계 구조를 명시하여 추론

중에서 택1



Self-Consistency CoT 적용

한 번만 추론하는 대신, **여러 개의 추론 경로** 를 샘플링한 뒤
각 경로가 내놓은 최종 정답을 모아 **다수결** 로 최종 답을 결정한다.
개별 경로는 실수할 수 있지만, 다수의 경로가 수렴하는 답이 더 신뢰할 수 있다는 아이디어.

In [25]:
import re
from collections import Counter

SC_SYSTEM = (
    "당신은 논리적 추론과 계산에 매우 정확한 한국어 AI입니다. "
    "문제의 조건을 하나씩 정리하고 계산 과정을 단계적으로 보인 뒤, "
    "맨 마지막 줄에 반드시 지정된 형식 그대로 '최종 정답:'을 적으세요. 형식을 절대 바꾸지 마세요."
)


def _final_line(text):
    """'최종 정답:' 뒤 문자열(없으면 마지막 비어있지 않은 줄)을 반환."""
    m = re.search(r"최종\s*정답\s*[:：]\s*(.+)", text)
    if m:
        return m.group(1).strip()
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[-1] if lines else text.strip()


def normalize_q1(text):
    """문제 1 전용: 최종 줄에서 (지불액, 포인트) 두 금액을 뽑아 표준 형식으로 통일."""
    nums = [int(n.replace(",", "")) for n in re.findall(r"\d[\d,]*", _final_line(text))]
    if len(nums) < 2:
        return None
    pay, point = nums[0], nums[1]
    return f"실제 지불 금액: {pay:,}원, 적립된 포인트: {point:,}원"


def normalize_q2(text):
    """문제 2 전용: 최종 줄의 확률을 n% 형식으로 통일(%, 분수, 소수 모두 대응)."""
    line = _final_line(text)
    m = re.search(r"(\d+(?:\.\d+)?)\s*%", line)          # 75%, 75.0%
    if m:
        return f"{float(m.group(1)):g}%"
    m = re.search(r"(\d+)\s*/\s*(\d+)", line)            # 3/4
    if m:
        return f"{int(m.group(1)) / int(m.group(2)) * 100:g}%"
    m = re.search(r"0?\.\d+", line)                       # 0.75
    if m:
        return f"{float(m.group(0)) * 100:g}%"
    return None


def self_consistency(question, answer_format, normalize,
                     n_paths=10, temperature=0.6, top_p=0.9,
                     max_new_tokens=768, verbose=True):
    """여러 추론 경로를 샘플링하고, 문제별 형식으로 정규화한 최종 정답을 다수결로 결정."""
    # 질문 + 문제별 출력 형식 지정
    user = question
    user += (
        "\n\n풀이 과정을 단계적으로 보인 뒤, 맨 마지막 줄에 반드시 아래 형식 그대로 답하세요.\n"
        f"최종 정답: {answer_format}"
    )

    messages = [
        {"role": "system", "content": SC_SYSTEM},
        {"role": "user", "content": user},
    ]

    # instruct 함수는 외부(HuggingFace pipeline 등)에서 정의되어 있다고 가정합니다.
    outs = instruct(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        num_return_sequences=n_paths,
    )

    # 각 추론 경로의 텍스트 → 문제별 형식으로 정규화한 최종 답
    paths = [o["generated_text"][-1]["content"] for o in outs]
    answers = [normalize(p) for p in paths]

    # 형식을 지킨(파싱 성공) 답만 다수결 집계
    vote = Counter(a for a in answers if a)
    final, count = vote.most_common(1)[0] if vote else ("(형식 파싱 실패)", 0)

    if verbose:
        print(f"총 {n_paths}개의 추론 경로를 샘플링했습니다.\n")
        # 전체 과정(p)은 출력하지 않고, 정규화된 최종결과만 출력
        for i, a in enumerate(answers, 1):
            print(f"추론 {i}번째 최종결과 : {a}")

        print("\n" + "=" * 60)
        print(f"투표 집계: {dict(vote)}")
        print(f"종합 최종결과 ({count}/{n_paths}표) : {final}")

    return final, paths, answers

In [26]:
# 문제 1
final_q1, paths_q1, answers_q1 = self_consistency(
    Q1,
    "실제 지불 금액: OOO원, 적립된 포인트: OOO원",
    normalize_q1,
    n_paths=10,
)

[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


총 10개의 추론 경로를 샘플링했습니다.

추론 1번째 최종결과 : 실제 지불 금액: 181,800원, 적립된 포인트: 9,090원
추론 2번째 최종결과 : 실제 지불 금액: 163,800원, 적립된 포인트: 8,190원
추론 3번째 최종결과 : 실제 지불 금액: 138,600원, 적립된 포인트: 6,930원
추론 4번째 최종결과 : 실제 지불 금액: 187,200원, 적립된 포인트: 9,360원
추론 5번째 최종결과 : 실제 지불 금액: 226,800원, 적립된 포인트: 11,340원
추론 6번째 최종결과 : 실제 지불 금액: 198,000원, 적립된 포인트: 9,900원
추론 7번째 최종결과 : 실제 지불 금액: 27원, 적립된 포인트: 309,600원
추론 8번째 최종결과 : 실제 지불 금액: 181,800원, 적립된 포인트: 9,090원
추론 9번째 최종결과 : 실제 지불 금액: 201,780원, 적립된 포인트: 10,620원
추론 10번째 최종결과 : 실제 지불 금액: 192,600원, 적립된 포인트: 9,630원

투표 집계: {'실제 지불 금액: 181,800원, 적립된 포인트: 9,090원': 2, '실제 지불 금액: 163,800원, 적립된 포인트: 8,190원': 1, '실제 지불 금액: 138,600원, 적립된 포인트: 6,930원': 1, '실제 지불 금액: 187,200원, 적립된 포인트: 9,360원': 1, '실제 지불 금액: 226,800원, 적립된 포인트: 11,340원': 1, '실제 지불 금액: 198,000원, 적립된 포인트: 9,900원': 1, '실제 지불 금액: 27원, 적립된 포인트: 309,600원': 1, '실제 지불 금액: 201,780원, 적립된 포인트: 10,620원': 1, '실제 지불 금액: 192,600원, 적립된 포인트: 9,630원': 1}
종합 최종결과 (2/10표) : 실제 지불 금액: 181,800원, 적립된 포인트: 9,090원


In [28]:
# 문제 2
final_q2, paths_q2, answers_q2 = self_consistency(
    Q2,
    "OO% (예: 75%)",
    normalize_q2,
    n_paths=10,
)

[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


총 10개의 추론 경로를 샘플링했습니다.

추론 1번째 최종결과 : 50%
추론 2번째 최종결과 : 75%
추론 3번째 최종결과 : 75%
추론 4번째 최종결과 : 45%
추론 5번째 최종결과 : 75%
추론 6번째 최종결과 : 75%
추론 7번째 최종결과 : 60%
추론 8번째 최종결과 : 30%
추론 9번째 최종결과 : 60%
추론 10번째 최종결과 : 60%

투표 집계: {'50%': 1, '75%': 4, '45%': 1, '60%': 3, '30%': 1}
종합 최종결과 (4/10표) : 75%
